In [1]:
import pandas as pd
import numpy as np
import re

# Load raw data
df = pd.read_csv('data/raw/NASCAR 2017-2024 Full Race  Points Data - Cup.csv')  # or my scraped file
print(f"Loaded {len(df)} rows")
print(f"Original columns: {df.columns.tolist()}")

Loaded 11325 rows
Original columns: ['year', 'race_num', 'track', 'track_type', 'fin', 'start', 'car_num', 'driver', 'manu', 'team_name', 'laps', 'laps_led', 'status', 'points', 'stage_1', 'stage_2', 'stage_3_or_duel', 'stage_points']


In [4]:
# Create a column mapping for consistent naming
# Adjust based on your actual column names
column_mapping = {
    # Common variations -> Standard names
    'Fin': 'Finish_Position',
    'fin': 'Finish_Position',
    'Pos': 'Finish_Position',
    'Position': 'Finish_Position',
    'St': 'Start_Position',
    'Start': 'Start_Position',
    'Driver': 'Driver',
    'Team': 'Team',
    'Owner': 'Team',
    'Laps': 'Laps_Completed',
    'Led': 'Laps_Led',
    'Laps Led': 'Laps_Led',
    'Laps_Led': 'Laps_Led',
    'Race': 'Race_Name',
    'Track': 'Track',
    'Date': 'Race_Date'
}

# Rename columns that exist in our mapping
df = df.rename(columns={col: column_mapping[col]
                        for col in df.columns
                        if col in column_mapping})

print(f"Standardized columns: {df.columns.tolist()}")

Standardized columns: ['year', 'race_num', 'track', 'track_type', 'Finish_Position', 'start', 'car_num', 'driver', 'manu', 'team_name', 'laps', 'laps_led', 'status', 'points', 'stage_1', 'stage_2', 'stage_3_or_duel', 'stage_points']


In [5]:
# Convert Finish_Position to numeric
# Handle non-numeric values (DNF, DNS, DQ may appear as text)
def clean_position(pos):
    """Convert position to integer, handling special cases."""
    if pd.isna(pos):
        return np.nan
    pos_str = str(pos).strip()
    # Extract just the number
    match = re.search(r'\d+', pos_str)
    if match:
        return int(match.group())
    return np.nan

df['Finish_Position'] = df['Finish_Position'].apply(clean_position)

# Same for Start_Position
if 'Start_Position' in df.columns:
    df['Start_Position'] = df['Start_Position'].apply(clean_position)

# Convert Laps_Led to numeric
if 'Laps_Led' in df.columns:
    df['Laps_Led'] = pd.to_numeric(df['Laps_Led'], errors='coerce').fillna(0).astype(int)

# Convert Race_Date to datetime
if 'Race_Date' in df.columns:
    df['Race_Date'] = pd.to_datetime(df['Race_Date'], errors='coerce')

print("\nData types after conversion:")
print(df.dtypes)


Data types after conversion:
year                 int64
race_num             int64
track                  str
track_type             str
Finish_Position      int64
start                int64
car_num              int64
driver                 str
manu                   str
team_name              str
laps                 int64
laps_led             int64
status                 str
points             float64
stage_1            float64
stage_2            float64
stage_3_or_duel    float64
stage_points           str
dtype: object


In [7]:
# View unique driver names to identify inconsistencies
print("Unique drivers:", df['driver'].nunique())
print("\nSample driver names:")
for name in sorted(df['driver'].unique())[:20]:
    print(f"  '{name}'")

Unique drivers: 154

Sample driver names:
  'A.J. Allmendinger'
  'Alex Bowman'
  'Alon Day'
  'Andy Lally'
  'Andy Seuss'
  'Anthony Alfredo'
  'Aric Almirola'
  'Austin Cindric'
  'Austin Dillon'
  'Austin Hill'
  'Austin Theriault'
  'B.J. McLeod'
  'Bayley Currey'
  'Ben Rhodes'
  'Billy Johnson'
  'Blake Jones'
  'Boris Said'
  'Brad Keselowski'
  'Brendan Gaughan'
  'Brennan Poole'


In [9]:
def clean_driver_name(name):
    """Standardize driver name format."""
    if pd.isna(name):
        return name

    # Remove extra whitespace
    name = ' '.join(str(name).split())

    # Remove common suffixes/prefixes
    name = re.sub(r'\s*\(i\)|\s*\*|\s*#\d+', '', name)

    # Capitalize properly
    name = name.title()

    return name.strip()

df['driver'] = df['driver'].apply(clean_driver_name)

# Check for common variations and standardize
driver_corrections = {
    # Add corrections as you find them, e.g.:
    # 'Denny M Hamlin': 'Denny Hamlin',
    # 'Hamlin, Denny': 'Denny Hamlin',
}

df['driver'] = df['driver'].replace(driver_corrections)

print(f"\nUnique drivers after cleaning: {df['driver'].nunique()}")


Unique drivers after cleaning: 154


In [12]:
def clean_team_name(name):
    """Standardize team name format."""
    if pd.isna(name):
        return name

    name = ' '.join(str(name).split())

    # Common standardizations
    corrections = {
        'Joe Gibbs Racing': 'Joe Gibbs Racing',
        'JGR': 'Joe Gibbs Racing',
        'Hendrick Motorsports': 'Hendrick Motorsports',
        'HMS': 'Hendrick Motorsports',
        'Team Penske': 'Team Penske',
        'Penske': 'Team Penske',
        'Stewart-Haas Racing': 'Stewart-Haas Racing',
        'SHR': 'Stewart-Haas Racing',
        '23XI Racing': '23XI Racing',
        '23XI': '23XI Racing',
        'Richard Childress Racing': 'Richard Childress Racing',
        'RCR': 'Richard Childress Racing',
        'Trackhouse Racing': 'Trackhouse Racing',
    }

    for old, new in corrections.items():
        if old.lower() in name.lower():
            return new

    return name.strip()

df['team_name'] = df['team_name'].apply(clean_team_name)

print(f"Unique teams: {df['team_name'].nunique()}")
print("\nTeam list:")
for team in sorted(df['team_name'].unique()):
    print(f"  {team}")

Unique teams: 41

Team list:
  23XI Racing
  B.J. McLeod Motorsports
  BK Racing
  Beard Motorsports
  Chip Ganassi Racing
  Circle Sport - The Motorsports Group
  Front Row Motorsports
  Furniture Row Racing
  Gaunt Brothers Racing
  Germain Racing
  Go Fas Racing
  Hendrick Motorsports
  JTG Daugherty Racing
  Joe Gibbs Racing
  Kaulig Racing
  Leavine Family Racing
  Legacy Motor Club
  Live Fast Motorsports
  MBM Motorsports
  NY Racing Team
  Obaika Racing
  Petty GMS Motorsports
  Premium Motorsports
  RBR Enterprises
  RFK Racing
  Richard Childress Racing
  Richard Petty Motorsports
  Rick Ware Racing
  Roush Fenway Racing
  Spire Motorsports
  StarCom Racing
  Stewart-Haas Racing
  Team AmeriVet
  Team Hezeberg
  Team Penske
  The Money Team Racing
  Tommy Baldwin Racing
  Trackhouse Racing
  TriStar Motorsports
  Wood Brothers Racing
  XCI Racing


In [14]:
# Create sponsor mapping for your target sponsors
# This maps driver+team combinations to their primary sponsor
# Based on your Project 1 selections

sponsor_mapping = {
    # Format: ('Driver Name', 'Team Name'): 'Primary Sponsor'

    # Example mappings for major 2024 sponsors:
    ('Denny Hamlin', 'Joe Gibbs Racing'): 'FedEx',
    ('Martin Truex Jr.', 'Joe Gibbs Racing'): 'Bass Pro Shops',
    ('Christopher Bell', 'Joe Gibbs Racing'): 'Rheem',
    ('Ty Gibbs', 'Joe Gibbs Racing'): 'Monster Energy',

    ('Kyle Larson', 'Hendrick Motorsports'): 'HendrickCars.com',
    ('Chase Elliott', 'Hendrick Motorsports'): 'NAPA Auto Parts',
    ('William Byron', 'Hendrick Motorsports'): 'Liberty University',
    ('Alex Bowman', 'Hendrick Motorsports'): 'Ally',

    ('Ryan Blaney', 'Team Penske'): 'Menards',
    ('Joey Logano', 'Team Penske'): 'Shell/Pennzoil',
    ('Austin Cindric', 'Team Penske'): 'Discount Tire',

    ('Bubba Wallace', '23XI Racing'): 'McDonald\'s',
    ('Tyler Reddick', '23XI Racing'): 'Jordan Brand',

    ('Ross Chastain', 'Trackhouse Racing'): 'Busch Light',
    ('Daniel Suarez', 'Trackhouse Racing'): 'Freeway Insurance',

    # Add more as needed based on your target sponsors
}

def get_sponsor(row):
    """Look up primary sponsor for driver/team combination."""
    key = (row['driver'], row['team_name'])
    return sponsor_mapping.get(key, 'Other/Unknown')

df['Primary_Sponsor'] = df.apply(get_sponsor, axis=1)

# Check sponsor distribution
print("Sponsor distribution:")
print(df['Primary_Sponsor'].value_counts())

Sponsor distribution:
Primary_Sponsor
Other/Unknown         8707
Shell/Pennzoil         296
FedEx                  296
NAPA Auto Parts        289
Menards                259
Liberty University     259
Ally                   251
Rheem                  148
Freeway Insurance      148
HendrickCars.com       147
McDonald's             147
Discount Tire          119
Busch Light            111
Monster Energy          74
Jordan Brand            74
Name: count, dtype: int64


In [15]:
# Define your target sponsors
target_sponsors = [
    'FedEx',
    'NAPA Auto Parts',
    'McDonald\'s',
    'Ally',
    'Busch Light'
    # just an example
]

# Create filtered dataset with just target sponsors
df_targets = df[df['Primary_Sponsor'].isin(target_sponsors)].copy()

print(f"\nFiltered to target sponsors: {len(df_targets)} rows")
print(df_targets['Primary_Sponsor'].value_counts())


Filtered to target sponsors: 1094 rows
Primary_Sponsor
FedEx              296
NAPA Auto Parts    289
Ally               251
McDonald's         147
Busch Light        111
Name: count, dtype: int64


In [19]:
# Add race number for time series analysis
races_order = df.groupby('race_num')['year'].min().sort_values()
race_number_map = {race: i+1 for i, race in enumerate(races_order.index)}
df['race_num'] = df['race_num'].map(race_number_map)

# Calculate positions gained/lost
if 'Start_Position' in df.columns:
    df['Positions_Gained'] = df['start'] - df['Finish_Position']

print("Added derived columns:")
print(df[['Race_Name', 'Race_Number', 'Driver', 'Start_Position',
          'Finish_Position', 'Positions_Gained']].head(10))

Added derived columns:


KeyError: "['Race_Name', 'Race_Number', 'Driver', 'Start_Position', 'Positions_Gained'] not in index"